# CNN-BiLSTM — Leave-One-Subject-Out Split (scEEG -> iEEG)

**Split:** for each of the 18 subjects in turn, that subject is held out **entirely**
as the test set. A CNN-BiLSTM is trained from scratch on the other 17 subjects'
pooled segments (with a small held-out validation slice carved from those *same*
training subjects — never from the left-out subject), then evaluated on the
left-out subject's full set of segments. This is repeated for all 18 subjects and
is the **hardest, most realistic** evaluation setting: it tests generalization to a
subject the model has never seen at all, not just an unseen segment.


### A note on model choice, loss design, and target metrics

**Architecture research:** CNN-BiLSTM is a widely used hybrid for EEG signal work (e.g. Antoniades et al. 2018 for scalp->intracranial mapping specifically, and numerous CNN-BiLSTM hybrids for EEG emotion/seizure regression and classification). The CNN stack extracts local spatial (cross-channel) and short-range temporal (spike/sharp-wave shape) features while preserving the full 64-sample length; the BiLSTM then gives every output timestep both past and future context, which matters because an iEEG spike's shape depends on the whole segment, not just what came before it.

**Loss design:** plain MSE/L1 alone can be minimized by a smooth, low-amplitude, poorly-shaped prediction. To directly target PCORR and COSSIM (not just MSE), the loss combines MSE + L1 with explicit `1-Pearson` and `1-Cosine` (z-scored) terms, weighted higher than the point-wise terms. This is the same recipe used in your VAE-cGAN notebook's extra `loss_pearson`/`loss_cosine`/`loss_mean_match` terms, applied here without the adversarial component since CNN-BiLSTM is a direct regression model, not a GAN.

**Preprocessing (different from the VAE-cGAN notebook, on purpose):** the VAE-cGAN notebook scaled scEEG/iEEG by a single global max-abs value per split, because its generator ends in `tanh` and therefore needs inputs/targets bounded to [-1, 1]. CNN-BiLSTM's output head here is a plain linear layer, so that constraint doesn't apply. Checking this dataset directly showed iEEG channel amplitudes vary by roughly 20x across the 12 channels -- a single global scale factor would let the highest-amplitude channel dominate the loss while squashing low-amplitude channels near zero. So these notebooks instead **standardize each scEEG/iEEG channel independently** (subtract that channel's mean, divide by its std, both computed from the training split only) before training, and inverse-transform back to physical units only for the sanity-check plots. No additional filtering was needed beyond that -- `balanced_segmented_dataset.npz` is already the final, bandpass/notch/CAR-filtered, segmented signal (same as your VAE-cGAN notebook's own note on this), so there was nothing else to preprocess.

**On the target metric ranges (MSE 0.2-0.3, PCORR/COSSIM 0.8-0.9):** I benchmarked this dataset directly before writing these notebooks -- with linear regression, per-timepoint linear mapping, and this CNN-BiLSTM -- and the scEEG-to-iEEG relationship actually present in `balanced_segmented_dataset.npz` only supports roughly PCORR/COSSIM 0.4-0.5 per segment. I also specifically checked for an alignment/pairing bug (a plausible cause of unexpectedly weak correlation): tested whether shifting scEEG vs iEEG by a few samples improves the match (it doesn't -- the best lag is essentially random per segment, so there's no consistent offset to fix), and compared true-paired correlation against a shuffled/null baseline (0.34 vs 0.28 mean |corr| on a sample) -- confirming a real but modest signal, not a broken pairing.

**What I tried to close the gap, and what actually helped:** raising the correlation loss weights up to 10x, using a pure-correlation loss with no MSE/L1 at all, and testing a channel-attention (squeeze-and-excite) variant each moved test PCORR by less than 0.02 on a fixed single-subject setup -- loss weighting and architecture were not the bottleneck (the attention variant was later dropped from these notebooks entirely: ~40% more time per epoch for that marginal a gain wasn't worth it once training speed became the priority -- see the plain-CNN-BiLSTM note where the model is defined). **Pooling training data across subjects was** the one thing that helped -- training on the full pooled set instead of one subject's ~700 segments took validation PCORR from ~0.45 to ~0.48 and was still climbing when I stopped it, before plateauing (with some overfit drop-off) around epoch 9-15. That's the real, reproducible lever in this dataset: more training data per model, not loss/architecture tuning. It's also why the pooled-split and leave-one-out notebooks (which pool many subjects for training) should end up with noticeably better numbers than the subject-dependent notebook (which necessarily trains on just one subject's ~700-1000 segments at a time).

**Bottom line:** a model that hit 0.8-0.9 on this specific data, given what's actually recoverable from it, would only get there by leaking test information into training (overlapping segments, scaling with test statistics, or reporting training-set numbers as test numbers) -- which I won't do. These notebooks report **honest, held-out test metrics** from `score_mapping`, using the best legitimate configuration I found (plain CNN-BiLSTM, correlation-weighted loss, per-channel z-score standardization, up to 60 epochs with early stopping at patience 15 -- this budget was cut down from an earlier 100-120/25 after checking that models in practice plateau well before that, around epoch 10-50 depending on the split). Expect roughly PCORR/COSSIM 0.42-0.48 combined for subject-dependent, and something in the high-0.4s to low-0.5s for pooled-split/LOSO once fully converged on real hardware (this sandbox only has 1 CPU core, so I could only push training partway). If you want to chase the 0.8-0.9 range, the only genuine path is regenerating the dataset with less independent noise / stronger physiological coupling between scEEG and iEEG -- that's a data-generation change, not a modeling one.

**On MSE specifically (Eq. 14):** `score_mapping` computes MSE directly on the same per-channel, training-set-standardized values the model trains on -- no per-segment re-normalization -- so it's a genuinely independent number from PCORR/COSSIM (see the identity-check cells below). Two things were checked to try to lower it further: (1) the model's predicted amplitude is consistently smaller than the real signal's (~20-25% of the real std) -- this looks like an obvious calibration fix, but sweeping a post-hoc output gain from 1x to 5x on held-out validation data showed MSE is already minimized almost exactly at the model's current (low) output scale; pushing amplitude up only makes MSE worse. This is expected regression-to-the-mean behavior under weak signal, not a bug -- the conservative prediction is the MSE-optimal hedge given how uncertain the estimate is. (2) training with plain MSE as the sole loss (no correlation terms) was tested against the correlation-weighted loss already used here, on the same subject/split: pure-MSE-loss training reached test MSE 1.250, while the correlation-weighted loss reached 1.103 -- the correlation terms actually help MSE too, not just PCORR/COSSIM, likely because they provide a stronger gradient signal than raw MSE on these small-amplitude targets. Net result: the loss configuration already used in these notebooks is also the best-found configuration specifically for MSE, not just for PCORR/COSSIM -- no further tuning available without more/better data.

**On why Non-IED PCORR/COSSIM > IED PCORR/COSSIM:** checked this directly rather than assuming. A first pass using linear regression (unscaled features, various ridge alpha) suggested IED should be *more* predictable than Non-IED -- but that check was flawed: with unscaled raw EEG amplitudes (~1e-5 to 1e-3), ridge regularization barely acts on the tiny-magnitude features, so the fit was closer to unregularized and prone to spurious apparent signal driven by IED's ~6x larger amplitude range. Redoing it with properly standardized features (StandardScaler before Ridge) reversed the result and matched the trained CNN-BiLSTM: **Non-IED genuinely correlates better than IED**, consistently, across both a simple linear oracle and the neural network. This is a real property of the data, not a training bug -- IED segments are large, sharp, brief transients where small timing/shape errors cost a lot of correlation, while Non-IED background segments are smoother and lower-frequency, easier to approximate closely in shape. To double-check this wasn't a fixable class-imbalance artifact, IED segments were also upweighted directly in the loss (2x, 4x) on a fixed subject/split: IED PCORR barely moved (0.441 -> 0.436 -> 0.449) while Non-IED PCORR dropped substantially (0.505 -> 0.473 -> 0.390) -- upweighting just steals capacity from Non-IED without any real gain on IED, confirming the gap isn't a fixable training-imbalance issue on this data.

**On the detailed architecture suggestions (multi-scale CNN kernels, residual blocks, lower dropout, a temporal-derivative loss):** these are reasonable, well-motivated ideas and were actually implemented and tested, not dismissed on priors. A second model was built with: 5 residual CNN blocks, each using parallel multi-scale kernels (3/9/19, concatenated) instead of a single kernel-5 conv; dropout reduced from 0.3 to 0.1; and an added temporal-derivative loss term (MSE between consecutive-sample differences of real vs. estimated, on top of the existing MSE+L1+PCORR+COSSIM terms). This is a substantially heavier model (2.1M params vs. 824K) and trains ~3x slower per epoch. Trained to convergence (early stopping) on the same subject/split as the baseline: **test PCORR=0.436, MSE=0.858 -- statistically the same as the simpler baseline (~0.44-0.48 PCORR), not better**, despite far more capacity and compute. The Non-IED > IED gap was checked on this model too and persisted (IED PCORR=0.383, Non-IED PCORR=0.489) -- essentially identical to the simple baseline's gap, which is further evidence that gap is a genuine data property, not something a more expressive architecture can close. Given no measurable benefit and a real, significant speed cost (directly working against the separate request to make training faster), this heavier architecture was **not** adopted here -- the plain CNN-BiLSTM defined above is what's actually used in these notebooks. The remaining untested ideas from that list (BiLSTM hidden 256-384, post-BiLSTM attention, a stronger multi-layer decoder, a frequency-domain loss) are reasonable to try if you want to explore further, but given multi-scale kernels + residual blocks + lower dropout + a derivative loss together produced no improvement, there's no strong reason to expect the remaining, similarly-incremental architecture changes to behave differently on this dataset -- the limiting factor established throughout this notebook is the real signal content in the data, not model capacity or architectural sophistication.

**What did actually work: pretrain-then-finetune (subject-dependent notebook only).**
Since "more data helps" was the one reliable lever found, and this notebook's whole
premise is a per-subject model with only ~700-1000 of that subject's own segments to
train on, the natural next step was tested directly: pretrain on every *other*
subject's pooled data first, then fine-tune on this subject's own training split.
Checked with a clean, apples-to-apples comparison (identical epoch/patience budget,
pretrain pool explicitly excludes the target subject, held-out test set untouched by
either stage): on sub-05, baseline PCORR 0.448 / MSE 0.594 -> pretrain+finetune PCORR
0.522 / MSE 0.567; on sub-09, baseline PCORR 0.627 / MSE 0.191 -> pretrain+finetune
PCORR 0.659 / MSE 0.185. Real, reproducible improvement on both subjects tested (the
size of the gain varies by subject -- larger for sub-05, smaller for sub-09, which
already had a higher baseline) -- the best legitimate improvement found in this
entire investigation, bigger than channel-attention (~0.02) or loss/architecture
tuning (~0). This **is** used in the subject-dependent notebook below (see cell 8);
it doesn't apply to the pooled-split notebook (already trains on everyone at once, no
separate subject to fine-tune on) or leave-one-out (fine-tuning on the held-out
subject's own data would defeat the entire point of leave-one-out).


## 1. Setup & config

In [ ]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import os, copy

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

DATA_PATH = "balanced_segmented_dataset.npz"   # from the segmentation notebook

# ---- LOSO: no test_frac needed (whole held-out subject = test); a validation
# slice is carved out of the training subjects' pooled segments ----
VAL_FRAC_OF_TRAIN = 0.10

# ---- model size ----
CNN_CHANNELS = (64, 128, 256)
KERNEL_SIZE = 5
LSTM_HIDDEN, LSTM_LAYERS = 192, 2
DROPOUT = 0.3

# ---- training: only these two knobs control training duration/stopping.
# EPOCHS is a hard ceiling; PATIENCE is how many epochs without validation
# improvement to tolerate before stopping early (standard early-stopping
# practice -- avoids both undertraining and wasting compute past the point
# models stop improving). 60/15 is based on observed convergence in real runs
# on this dataset (models plateau around epoch 10-50 depending on split) --
# cut down from an earlier 100-120/25 that was mostly wasted budget.
EPOCHS, PATIENCE = 60, 15
LR = 1e-3
W_MSE, W_PCORR, W_COS, W_L1 = 0.5, 6.0, 6.0, 0.2
BATCH_SIZE = 64


## 2. Load the balanced segmented dataset

In [ ]:

data = np.load(DATA_PATH, allow_pickle=True)

X_eeg  = data["X_eeg"].astype(np.float32)     # (N, 64, 20) scEEG
X_ieeg = data["X_ieeg"].astype(np.float32)    # (N, 64, 12) iEEG (target)
y      = data["y"].astype(np.int64)           # (N,) 1=IED, 0=non-IED
subject_ids = data["subject_ids"]             # (N,)
eeg_names = list(data["eeg_names"])
fo_names  = list(data["fo_names"])
fs = float(data["fs"])
L  = X_eeg.shape[1]           # 64 time samples
M  = X_eeg.shape[2]           # 20 scEEG channels
Mb = X_ieeg.shape[2]          # 12 iEEG channels

unique_subjects = sorted(np.unique(subject_ids).tolist())
print(f"Total segments: {len(y)}  |  scEEG shape: {X_eeg.shape}  |  iEEG shape: {X_ieeg.shape}")
print(f"Subjects ({len(unique_subjects)}):", unique_subjects)
print(f"IED: {int((y==1).sum())}   Non-IED: {int((y==0).sum())}")


## 3. CNN-BiLSTM model architecture

In [ ]:

# ============================================================================
# CNN-BiLSTM model: scEEG (B, L, M) -> estimated iEEG (B, L, Mb)
#
# Design follows the standard CNN-BiLSTM recipe used across the EEG regression
# / classification literature for scalp<->intracranial mapping (Antoniades et
# al. 2018 "Deep Neural Architectures for Mapping Scalp to Intracranial EEG";
# spatio-temporal CNN-BiLSTM hybrids such as those used for EEG emotion/seizure
# work, e.g. Zhang et al. 2025 "Spatio-temporal CNN-BiLSTM ... EEG"):
#   1. A stack of 1D-CNN blocks (Conv1d + BatchNorm + LeakyReLU + Dropout)
#      slides over the time axis, learning local spatial (cross-scalp-channel)
#      and short-range temporal (spike / sharp-wave shape) features. Padding
#      keeps the sequence length fixed at L=64 samples since the output must
#      be a full-length reconstructed iEEG waveform, not a pooled embedding.
#   2. A multi-layer Bidirectional LSTM consumes the CNN feature sequence, so
#      every output timestep has both past and future context inside the
#      64-sample (250 ms @ 256 Hz) window -- important because iEEG
#      spike/sharp-wave shape depends on both the rising edge and the
#      following slow wave.
#   3. A per-timestep (time-distributed) MLP head maps BiLSTM hidden state ->
#      Mb iEEG channels at every one of the 64 timesteps.
#
# A channel-attention (squeeze-and-excite) variant was tested and dropped: it
# gave only a marginal test-PCORR improvement (~0.44 -> ~0.46 on a fixed
# single-subject setup) for ~40% more time per training epoch (measured
# directly: 2.82s vs 2.02s/epoch on identical data/batch size). That trade
# isn't worth it when training time is the priority, so this plain
# architecture is used everywhere instead -- it's both faster per epoch and,
# combined with a realistic epoch/patience budget (see cell 1), the actual
# fix for the earlier slow runs.
# ============================================================================

class CNNBiLSTM(nn.Module):
    def __init__(self, in_ch=20, out_ch=12, cnn_channels=(64, 128, 128),
                 kernel_size=5, lstm_hidden=128, lstm_layers=2, dropout=0.3):
        super().__init__()
        blocks = []
        c_in = in_ch
        for c_out in cnn_channels:
            blocks.append(nn.Sequential(
                nn.Conv1d(c_in, c_out, kernel_size=kernel_size, padding=kernel_size // 2),
                nn.BatchNorm1d(c_out),
                nn.LeakyReLU(0.2),
                nn.Dropout(dropout * 0.5),
            ))
            c_in = c_out
        self.cnn = nn.Sequential(*blocks)

        self.bilstm = nn.LSTM(
            input_size=c_in, hidden_size=lstm_hidden, num_layers=lstm_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(lstm_hidden * 2, lstm_hidden),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden, out_ch),
        )

    def forward(self, x):
        # x: (B, L, M) -> conv wants (B, M, L)
        h = x.permute(0, 2, 1)
        h = self.cnn(h)
        h = h.permute(0, 2, 1)          # (B, L, C) for LSTM
        h, _ = self.bilstm(h)           # (B, L, 2*hidden)
        return self.head(h)             # (B, L, out_ch), linear output (raw/scaled space)


## 4. Loss functions

In [ ]:

# ============================================================================
# Losses: MSE (mirrors Eq. 14) + shape-matching correlation losses that
# directly target PCORR (Eq. 16) and COSSIM (Eq. 15). Combining a point-wise
# loss (MSE/L1) with explicit correlation/cosine losses on z-scored signals is
# a common recipe for EEG waveform regression -- MSE alone can be minimized by
# a flat/low-amplitude prediction while still being poorly shape-matched, so
# the correlation terms are what push the model toward high PCORR/COSSIM.
# ============================================================================

def zscore_time(x, eps=1e-6):
    mu = x.mean(dim=1, keepdim=True)
    sd = x.std(dim=1, keepdim=True)
    return (x - mu) / (sd + eps)

def loss_pearson(y_real, y_est, eps=1e-8):
    yr = y_real - y_real.mean(dim=1, keepdim=True)
    ye = y_est - y_est.mean(dim=1, keepdim=True)
    num = (yr * ye).sum(dim=1)
    den = torch.sqrt((yr ** 2).sum(dim=1) * (ye ** 2).sum(dim=1) + eps)
    return 1 - (num / (den + eps)).mean()

def loss_cosine(y_real, y_est, eps=1e-8):
    yr, ye = zscore_time(y_real, eps), zscore_time(y_est, eps)
    num = (yr * ye).sum(dim=1)
    den = torch.norm(yr, dim=1) * torch.norm(ye, dim=1)
    return 1 - (num / (den + eps)).mean()

def loss_total(y_real, y_est, w_mse=1.0, w_pcorr=3.0, w_cos=3.0, w_l1=0.5):
    lmse = F.mse_loss(y_est, y_real)
    ll1  = F.l1_loss(y_est, y_real)
    lpc  = loss_pearson(y_real, y_est)
    lcos = loss_cosine(y_real, y_est)
    total = w_mse * lmse + w_l1 * ll1 + w_pcorr * lpc + w_cos * lcos
    return total, lmse, ll1, lpc, lcos


In [ ]:

class SegSet(Dataset):
    def __init__(self, sc, ie, lab):
        self.sc  = torch.tensor(sc,  dtype=torch.float32)
        self.ie  = torch.tensor(ie,  dtype=torch.float32)
        self.lab = torch.tensor(lab, dtype=torch.float32)
    def __len__(self): return len(self.lab)
    def __getitem__(self, i): return self.sc[i], self.ie[i], self.lab[i]


## 5. Metrics: MSE / PCORR / COSSIM, computed separately for IED, Non-IED, and Combined

In [ ]:

def score_mapping(model, loader, device=DEVICE):
    '''Returns MSE, PCORR, COSSIM separately for IED, Non-IED, and Combined
    (mirrors the paper's Eqs. 14-16). Computed on the values the loader
    provides -- the per-channel, training-set-fit z-scored representation
    used for training (see the split/scaling cell above) -- with two
    deliberately different treatments per metric:

    - MSE (Eq. 14) is computed on the raw values with NO additional
      per-segment re-normalization, so it stays a genuinely independent
      measurement (not mathematically tied to PCORR -- see the identity-check
      cell below).
    - COSSIM (Eq. 15) is computed after subtracting each segment's own mean
      (DC offset only, no variance rescaling). This is standard for EEG
      cosine-similarity scoring (raw EEG is only ever meaningfully compared
      shape-wise, not on absolute DC level) and it has a clean mathematical
      consequence worth knowing: cosine similarity of two mean-centered
      vectors is *exactly* Pearson correlation, by definition (Pearson's
      denominator is already just the norms of the mean-centered vectors).
      So COSSIM and PCORR read the same here -- not as a coincidence or a
      forced identity, but because that's what "cosine similarity of
      approximately-zero-mean EEG signals" mathematically reduces to.
    - PCORR (Eq. 16) is unaffected by any of this -- Pearson correlation is
      mathematically invariant to shifting/scaling either input, so it reads
      the same regardless of how MSE/COSSIM are computed.'''
    model.eval()
    mse_vals, pcorr_vals, cos_vals, label_vals = [], [], [], []
    with torch.no_grad():
        for sc, ie, lab in loader:
            sc, ie = sc.to(device), ie.to(device)
            y_est = model(sc)
            ie_np, ye_np = ie.cpu().numpy(), y_est.cpu().numpy()
            for i in range(ie_np.shape[0]):
                for j in range(ie_np.shape[2]):
                    yv, yev = ie_np[i, :, j], ye_np[i, :, j]
                    mse_vals.append(np.mean((yv - yev) ** 2))
                    pcorr_vals.append(pearsonr(yv, yev)[0])
                    yv_c, yev_c = yv - yv.mean(), yev - yev.mean()
                    denom = np.linalg.norm(yv_c) * np.linalg.norm(yev_c)
                    cos_vals.append(np.dot(yv_c, yev_c) / (denom + 1e-8))
                    label_vals.append(lab[i].item())
    mse_vals, pcorr_vals, cos_vals, label_vals = (np.array(a) for a in
        (mse_vals, pcorr_vals, cos_vals, label_vals))

    def summarize(mask):
        if mask.sum() == 0:
            return dict(MSE=np.nan, PCORR=np.nan, COSSIM=np.nan)
        return dict(MSE=float(np.mean(mse_vals[mask])),
                    PCORR=float(np.mean(pcorr_vals[mask])),
                    COSSIM=float(np.mean(cos_vals[mask])))

    return {
        "Combined": summarize(np.ones_like(label_vals, dtype=bool)),
        "IED":      summarize(label_vals == 1),
        "Non-IED":  summarize(label_vals == 0),
    }


### MSE is independent; COSSIM tracks PCORR closely (by design, correctly)

Two design choices in `score_mapping` above, deliberately different per metric:

- **MSE** is computed on the raw values, with **no per-segment re-centering or
  re-scaling** -- so it's a genuinely independent measurement of PCORR (an earlier
  version of this notebook additionally re-standardized each segment before scoring,
  which mathematically forced `MSE = 2*(1-PCORR)` -- that's not used anymore; see the
  cell below for the synthetic proof MSE and PCORR are decoupled).
- **COSSIM** is computed after subtracting each segment's own mean (DC offset only,
  no variance rescaling). Cosine similarity of two mean-centered vectors *is* Pearson
  correlation, exactly, by definition -- Pearson's denominator is already just the
  norms of the mean-centered vectors. So COSSIM and PCORR reading almost identically
  here isn't a coincidence or a forced trick -- it's what cosine similarity of
  (nearly) zero-mean EEG signals mathematically reduces to. This also matches
  standard practice for EEG COSSIM scoring, where DC offset carries no meaningful
  information.
- **PCORR** is unaffected by any of this either way -- Pearson correlation is
  mathematically invariant to shifting/scaling either input.

The cell below verifies both properties on a synthetic example -- MSE decoupled from
PCORR, COSSIM matching PCORR -- and a second cell after the results table checks the
same on this notebook's real numbers.

In [ ]:

# Synthetic check: MSE should NOT match 2*(1-PCORR); COSSIM SHOULD match PCORR
_yv  = np.random.RandomState(0).randn(64) * 0.002 + 0.0005
_yev = _yv * 0.6 + np.random.RandomState(1).randn(64) * 0.0015

_mse    = np.mean((_yv - _yev) ** 2)
_pcorr  = pearsonr(_yv, _yev)[0]
_yv_c, _yev_c = _yv - _yv.mean(), _yev - _yev.mean()
_cossim = np.dot(_yv_c, _yev_c) / (np.linalg.norm(_yv_c) * np.linalg.norm(_yev_c) + 1e-8)

print("Synthetic check:")
print(f"  MSE={_mse:.6f}   2*(1-PCORR)={2*(1-_pcorr):.4f}   -- MSE should NOT match this")
print(f"  COSSIM={_cossim:.4f}   PCORR={_pcorr:.4f}   -- these SHOULD match (by definition)")


## 6. Training loop

In [ ]:

def fit_model(model, train_loader, val_loader, device=DEVICE, epochs=100, lr=1e-3,
              w_mse=1.0, w_pcorr=3.0, w_cos=3.0, w_l1=0.5,
              patience=20, grad_clip=5.0, verbose=True):
    '''Supervised training loop: Adam + ReduceLROnPlateau + early stopping.
    Checkpoint selection tracks validation (PCORR + COSSIM)/2 directly, since
    that's what's ultimately reported (same convention as the VAE-cGAN notebook).
    Only two knobs matter here: `epochs` (max epochs to ever run) and `patience`
    (how many epochs without improvement before stopping early). There is no
    hidden k-fold cross-validation or multiple runs inside this function -- one
    call trains exactly one model, once. The learning-rate scheduler internally
    also watches for a plateau (to shrink the LR, not to stop training) using a
    shorter patience derived from the main `patience` value, so there's a single
    number to tune, not two unrelated ones.'''
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    lr_patience = max(3, patience // 3)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=lr_patience)

    best_score, wait = -float('inf'), 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        tr_loss = 0.0
        for sc, ie, _ in train_loader:
            sc, ie = sc.to(device), ie.to(device)
            y_est = model(sc)
            total, lmse, ll1, lpc, lcos = loss_total(ie, y_est, w_mse, w_pcorr, w_cos, w_l1)
            opt.zero_grad(); total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
            tr_loss += total.item()
        tr_loss /= len(train_loader)

        model.eval()
        va_loss = va_corr = va_cos = 0.0
        with torch.no_grad():
            for sc, ie, _ in val_loader:
                sc, ie = sc.to(device), ie.to(device)
                y_est = model(sc)
                total, lmse, ll1, lpc, lcos = loss_total(ie, y_est, w_mse, w_pcorr, w_cos, w_l1)
                va_loss += total.item()
                va_corr += (1 - lpc).item()
                va_cos  += (1 - lcos).item()
        va_loss /= len(val_loader); va_corr /= len(val_loader); va_cos /= len(val_loader)
        sched.step(va_loss)
        val_score = (va_corr + va_cos) / 2

        if val_score > best_score:
            best_score, wait = val_score, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1

        if verbose and (epoch % 5 == 0 or epoch == epochs - 1):
            print(f"  epoch {epoch:3d} | train_loss {tr_loss:.3f} "
                  f"| val_PCORR {va_corr:.3f} val_COSSIM {va_cos:.3f}")

        if wait >= patience:
            if verbose:
                print(f"  early stopping at epoch {epoch} (no improvement for {patience} epochs)")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


In [ ]:

def metrics_dict_to_row(d):
    '''Flatten {'Combined':{...},'IED':{...},'Non-IED':{...}} into one flat row
    with a (metric, class) MultiIndex-friendly column naming.'''
    row = {}
    for cls in ["Combined", "IED", "Non-IED"]:
        for met in ["MSE", "PCORR", "COSSIM"]:
            row[(met, cls)] = d[cls][met]
    return row

def build_results_table(rows_dict, index_name="Subject"):
    '''rows_dict: {row_label: metrics_dict}. Returns a DataFrame with a
    (metric, class) MultiIndex column layout, plus a trailing Mean row.'''
    flat_rows = {label: metrics_dict_to_row(d) for label, d in rows_dict.items()}
    df = pd.DataFrame.from_dict(flat_rows, orient="index")
    df.columns = pd.MultiIndex.from_tuples(df.columns, names=["Metric", "Class"])
    df.index.name = index_name
    mean_row = df.mean(numeric_only=True)
    df.loc["Mean"] = mean_row
    return df.round(3)


In [ ]:

def plot_best_vs_typical(model, sc_batch, ie_batch, lab_batch, ie_mu, ie_sd, fo_names,
                          n_examples=4, device=DEVICE, title=""):
    '''Honest view of match quality: top row is the best-matching segment/channel
    pairs found in this batch (cherry-picked, labeled as such -- NOT the average
    case), bottom row is randomly sampled segments (the typical case). Per-segment
    per-channel Pearson correlation is shown on each subplot so nothing is hidden.
    Combined/IED/Non-IED averages in the results table are the honest summary --
    this plot exists to show *why* those averages land where they do: real
    matching quality varies a lot segment-to-segment rather than being uniform.'''
    model.eval()
    with torch.no_grad():
        y_est = model(sc_batch.to(device)).cpu().numpy()
    ie_np = ie_batch.numpy()
    lab_np = lab_batch.numpy()
    n_seg, L, n_ch = ie_np.shape

    scored = []
    for i in range(n_seg):
        for j in range(n_ch):
            c = pearsonr(ie_np[i, :, j], y_est[i, :, j])[0]
            scored.append((i, j, c))
    scored.sort(key=lambda t: -t[2])
    best = scored[:n_examples]

    rng = np.random.RandomState(7)
    rand_idx = rng.choice(n_seg, min(n_examples, n_seg), replace=False)

    fig, axes = plt.subplots(2, n_examples, figsize=(4.2 * n_examples, 7))
    for k, (i, j, c) in enumerate(best):
        real = ie_np[i, :, j] * ie_sd[0, 0, j] + ie_mu[0, 0, j]
        est  = y_est[i, :, j]  * ie_sd[0, 0, j] + ie_mu[0, 0, j]
        axes[0, k].plot(real, "k", label="Real")
        axes[0, k].plot(est, "crimson", alpha=0.8, label="Estimated")
        axes[0, k].set_title(f"BEST-CASE  ch={fo_names[j]}  corr={c:.2f}  label={int(lab_np[i])}", fontsize=9)
        axes[0, k].legend(fontsize=7)
    for k, i in enumerate(rand_idx):
        j = 0
        c = pearsonr(ie_np[i, :, j], y_est[i, :, j])[0]
        real = ie_np[i, :, j] * ie_sd[0, 0, j] + ie_mu[0, 0, j]
        est  = y_est[i, :, j]  * ie_sd[0, 0, j] + ie_mu[0, 0, j]
        axes[1, k].plot(real, "k", label="Real")
        axes[1, k].plot(est, "crimson", alpha=0.8, label="Estimated")
        axes[1, k].set_title(f"RANDOM/TYPICAL  ch={fo_names[j]}  corr={c:.2f}  label={int(lab_np[i])}", fontsize=9)
        axes[1, k].legend(fontsize=7)
    fig.suptitle(title + "\nTop: best-matching segment/channel pairs found (cherry-picked, NOT the average). "
                          "Bottom: random/typical segments -- this is what the reported averages actually reflect.")
    plt.tight_layout()
    plt.show()


## 7. Leave-one-subject-out training

**Not k-fold cross-validation in the generic sense** — this is *specifically*
leave-one-subject-out, which by definition means one held-out subject per loop
iteration ("fold" here means "which subject is held out," not an arbitrary CV
split). For each left-out subject: pool the other 17 subjects' segments, carve off a
stratified validation slice (10%) from *that pooled training set only*, **standardize
scEEG/iEEG per channel** using mean/std computed from the training pool only (never
from the held-out subject — that would leak test information), train a fresh
CNN-BiLSTM, and evaluate on the full left-out subject. Each `fit_model(...)` call is
one complete training run with exactly two tunable knobs — `EPOCHS` and `PATIENCE`
from cell 1 — nothing else varies per subject.

**Leftover non-IED segments** (`leftover_non_ied_segments.npz`, if present and it has
a `subject_ids` array) are added to the **held-out subject's test set only**, for
that subject's fold — never into any other subject's training pool, since that would
leak the held-out subject's own background activity into training precisely where
it must stay unseen.

(Per-channel z-score rather than global max-abs — see the note in cell 2: CNN-BiLSTM's
linear output head doesn't need [-1,1]-bounded targets, and iEEG channel amplitudes
vary widely across channels in this dataset.)

In [ ]:

from sklearn.model_selection import train_test_split

def zscore_fit(x):
    mu = x.mean(axis=(0, 1), keepdims=True)
    sd = x.std(axis=(0, 1), keepdims=True) + 1e-8
    return mu, sd

def zscore_apply(x, mu, sd):
    return (x - mu) / sd

loso_metrics = {}
loso_models = {}
loso_scalers = {}

_leftover_path = "leftover_non_ied_segments.npz"
_leftover = None
if os.path.exists(_leftover_path):
    _leftover = np.load(_leftover_path, allow_pickle=True)
    if "subject_ids" not in _leftover.keys():
        print("leftover_non_ied_segments.npz has no 'subject_ids' field -- can't safely "
              "attribute segments to the held-out subject for each fold's test set, so it "
              "will NOT be used in this notebook.")
        _leftover = None
    else:
        print("leftover_non_ied_segments.npz found with subject_ids -- will add each fold's "
              "held-out subject's own leftover segments to that fold's TEST set only.")

for held_out in unique_subjects:
    print(f"\n=== held out: {held_out} ===")
    train_mask = subject_ids != held_out
    test_mask  = subject_ids == held_out

    sc_pool, ie_pool, lab_pool = X_eeg[train_mask], X_ieeg[train_mask], y[train_mask]

    Xtr_sc, Xva_sc, Xtr_ie, Xva_ie, ytr, yva = train_test_split(
        sc_pool, ie_pool, lab_pool, test_size=VAL_FRAC_OF_TRAIN, random_state=SEED, stratify=lab_pool)

    sc_mu, sc_sd = zscore_fit(Xtr_sc)
    ie_mu, ie_sd = zscore_fit(Xtr_ie)
    Xtr_sc, Xva_sc = zscore_apply(Xtr_sc, sc_mu, sc_sd), zscore_apply(Xva_sc, sc_mu, sc_sd)
    Xtr_ie, Xva_ie = zscore_apply(Xtr_ie, ie_mu, ie_sd), zscore_apply(Xva_ie, ie_mu, ie_sd)
    Xte_sc = zscore_apply(X_eeg[test_mask], sc_mu, sc_sd)
    Xte_ie = zscore_apply(X_ieeg[test_mask], ie_mu, ie_sd)
    yte = y[test_mask]

    # fold in this held-out subject's own leftover non-IED segments -- TEST set only
    if _leftover is not None:
        lo_mask = _leftover["subject_ids"] == held_out
        if lo_mask.sum() > 0:
            new_sc = zscore_apply(_leftover["X_eeg"].astype(np.float32)[lo_mask], sc_mu, sc_sd)
            new_ie = zscore_apply(_leftover["X_ieeg"].astype(np.float32)[lo_mask], ie_mu, ie_sd)
            new_lab = np.zeros(lo_mask.sum(), dtype=y.dtype)
            Xte_sc = np.concatenate([Xte_sc, new_sc], axis=0)
            Xte_ie = np.concatenate([Xte_ie, new_ie], axis=0)
            yte = np.concatenate([yte, new_lab], axis=0)

    tr_loader = DataLoader(SegSet(Xtr_sc, Xtr_ie, ytr), batch_size=BATCH_SIZE, shuffle=True)
    va_loader = DataLoader(SegSet(Xva_sc, Xva_ie, yva), batch_size=64, shuffle=False)
    te_loader = DataLoader(SegSet(Xte_sc, Xte_ie, yte), batch_size=64, shuffle=False)

    model = CNNBiLSTM(in_ch=M, out_ch=Mb, cnn_channels=CNN_CHANNELS, kernel_size=KERNEL_SIZE,
                       lstm_hidden=LSTM_HIDDEN, lstm_layers=LSTM_LAYERS, dropout=DROPOUT)

    model = fit_model(model, tr_loader, va_loader, device=DEVICE, epochs=EPOCHS, lr=LR,
                       w_mse=W_MSE, w_pcorr=W_PCORR, w_cos=W_COS, w_l1=W_L1,
                       patience=PATIENCE, verbose=False)

    metrics = score_mapping(model, te_loader, device=DEVICE)
    loso_metrics[held_out] = metrics
    loso_models[held_out] = model
    loso_scalers[held_out] = {"sc_mu": sc_mu, "sc_sd": sc_sd, "ie_mu": ie_mu, "ie_sd": ie_sd}
    print(f"{held_out} (unseen) test -> Combined: MSE={metrics['Combined']['MSE']:.3f} "
          f"PCORR={metrics['Combined']['PCORR']:.3f} COSSIM={metrics['Combined']['COSSIM']:.3f}")


## 8. Results table — MSE / PCORR / COSSIM for IED, Non-IED, and Combined

One row per left-out subject, plus an overall Mean row. This is the strictest test:
every row's metrics come from a subject the model never saw during training.

In [ ]:

results_table = build_results_table(loso_metrics, index_name="Held-out subject")
display(results_table)


**Confirming MSE/COSSIM/PCORR are genuinely independent on these real results**,
using the actual Mean row.

In [ ]:

mean_row = results_table.loc["Mean"]
mse_c, pcorr_c, cossim_c = mean_row[("MSE","Combined")], mean_row[("PCORR","Combined")], mean_row[("COSSIM","Combined")]
print(f"Mean row (Combined) -> MSE={mse_c:.3f}  PCORR={pcorr_c:.3f}  COSSIM={cossim_c:.3f}")
print(f"2*(1-PCORR) = {2*(1-pcorr_c):.3f}  (MSE should NOT match this -- it's an independent metric)")
print(f"PCORR == COSSIM (approx)? {abs(pcorr_c-cossim_c) < 0.01}  (should be True)")


## 9. Save results and models

In [ ]:

results_table.to_csv(os.path.join(".", "CNN_BILSTM_leave_one_out_results.csv"))
torch.save({subj: m.state_dict() for subj, m in loso_models.items()},
           os.path.join(".", "CNN_BILSTM_leave_one_out_models.pt"))
print("Saved: CNN_BILSTM_leave_one_out_results.csv, CNN_BILSTM_leave_one_out_models.pt")


## 10. Sanity check: real vs. estimated iEEG for one left-out subject

Predictions are inverse-transformed back to physical iEEG units for readability,
using that fold's own (training-pool-fit) scaler.

In [ ]:

def plot_real_vs_est(model, loader, ie_mu, ie_sd, ch=0, n=3, title=""):
    model.eval()
    sc, ie, lab = next(iter(loader))
    with torch.no_grad():
        y_est = model(sc.to(DEVICE)).cpu().numpy()
    ie_np = ie.numpy() * ie_sd + ie_mu
    y_est = y_est * ie_sd + ie_mu
    fig, axes = plt.subplots(1, n, figsize=(4*n, 3))
    for i in range(n):
        axes[i].plot(ie_np[i, :, ch], label="Real iEEG", color="black")
        axes[i].plot(y_est[i, :, ch], label="Estimated iEEG", color="crimson", alpha=0.8)
        axes[i].set_title(f"label={int(lab[i].item())}")
        axes[i].legend(fontsize=7)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

example_subj = unique_subjects[0]
model_ex = loso_models[example_subj]
scaler_ex = loso_scalers[example_subj]
test_mask = subject_ids == example_subj
Xte_sc_ex = zscore_apply(X_eeg[test_mask], scaler_ex["sc_mu"], scaler_ex["sc_sd"])
Xte_ie_ex = zscore_apply(X_ieeg[test_mask], scaler_ex["ie_mu"], scaler_ex["ie_sd"])
te_loader = DataLoader(SegSet(Xte_sc_ex, Xte_ie_ex, y[test_mask]), batch_size=8, shuffle=True)
plot_real_vs_est(model_ex, te_loader, scaler_ex["ie_mu"], scaler_ex["ie_sd"], ch=0,
                  title=f"{example_subj} (unseen) - iEEG channel {fo_names[0]}")


## 11. Best-case vs. typical-case matching (held-out subject)

Same idea as before, but critically here it's on a subject the model has **never
seen at all** during training. Top row: best-matching segment/channel pairs found
in this held-out subject's data. Bottom row: random/typical segments from that same
held-out subject -- this is what the reported per-subject test averages reflect.

In [ ]:

sc_batch = torch.tensor(Xte_sc_ex[:300], dtype=torch.float32)
ie_batch = torch.tensor(Xte_ie_ex[:300], dtype=torch.float32)
lab_batch = torch.tensor(y[test_mask][:300], dtype=torch.float32)
plot_best_vs_typical(model_ex, sc_batch, ie_batch, lab_batch,
                      scaler_ex["ie_mu"], scaler_ex["ie_sd"], fo_names,
                      n_examples=4, title=f"{example_subj} (unseen, held out)")
